In [3]:
import os, re, hashlib, collections
from pathlib import Path
import yaml

FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_output")
assert FOLDER.exists(), f"Folder not found: {FOLDER.resolve()}"

yaml_files = sorted(FOLDER.glob("*.yaml")) + sorted(FOLDER.glob("*.yml"))
print(f"YAML files found: {len(yaml_files)}")

YAML files found: 2500


In [4]:
corrupt      = []
parse_errors = []
ok_count     = 0

for fp in yaml_files:
    try:
        raw = fp.read_text(encoding="utf-8", errors="replace")
        if not raw.strip():
            corrupt.append(fp.name)
            continue
        data = yaml.safe_load(raw)
        if not isinstance(data, dict):
            corrupt.append(fp.name)
            continue
        ok_count += 1
    except yaml.YAMLError as e:
        parse_errors.append((fp.name, str(e)[:80]))

print(f"OK           : {ok_count}")
print(f"Parse errors : {len(parse_errors)}")
print(f"Corrupt/empty: {len(corrupt)}")

OK           : 2500
Parse errors : 0
Corrupt/empty: 0


In [5]:
from pathlib import Path
import yaml

OUTPUT_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned")
OUTPUT_FOLDER.mkdir(exist_ok=True)

LIST_FIELDS = ["responsibilities", "basic_qualifications", "preferred_qualifications"]

def clean_job(data: dict) -> dict:
    cleaned = {}
    for key, val in data.items():

        if key in LIST_FIELDS:
            # ensure it's a list
            if val is None:
                val = []
            elif isinstance(val, str):
                val = [val]
            elif not isinstance(val, list):
                val = [str(val)]

            # clean each item: strip whitespace, drop empty/null
            items = []
            for item in val:
                if item is None:
                    continue
                item = str(item).strip()
                item = " ".join(item.split())  # collapse multiple spaces
                if item:
                    items.append(item)

            # remove exact duplicates, preserve order
            seen = set()
            deduped = []
            for item in items:
                if item not in seen:
                    seen.add(item)
                    deduped.append(item)

            cleaned[key] = deduped

        else:
            # scalar field — just normalize whitespace if string
            if isinstance(val, str):
                val = " ".join(val.strip().split())
            cleaned[key] = val

    return cleaned

# process one by one
saved    = 0
skipped  = 0

for fp in yaml_files:
    try:
        raw  = fp.read_text(encoding="utf-8", errors="replace")
        data = yaml.safe_load(raw)

        cleaned = clean_job(data)

        out_path = OUTPUT_FOLDER / fp.name
        with open(out_path, "w", encoding="utf-8") as f:
            yaml.dump(cleaned, f, allow_unicode=True, sort_keys=False, default_flow_style=False)

        saved += 1

    except Exception as e:
        print(f"SKIPPED {fp.name}: {e}")
        skipped += 1

print(f"Saved  : {saved}")
print(f"Skipped: {skipped}")
print(f"Output : {OUTPUT_FOLDER}")

Saved  : 2500
Skipped: 0
Output : C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned


In [6]:
import yaml
from pathlib import Path

ORIGINAL_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_output")
CLEANED_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned")

changed   = 0
unchanged = 0

for fp in ORIGINAL_FOLDER.glob("*.yaml"):
    orig    = yaml.safe_load(fp.read_text(encoding="utf-8", errors="replace"))
    cleaned = yaml.safe_load((CLEANED_FOLDER / fp.name).read_text(encoding="utf-8", errors="replace"))
    if orig != cleaned:
        changed += 1
    else:
        unchanged += 1

print(f"Changed  : {changed}")
print(f"Unchanged: {unchanged}")

Changed  : 1679
Unchanged: 821


In [7]:
import yaml
from pathlib import Path

ORIGINAL_FOLDER = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_output")
CLEANED_FOLDER  = Path(r"C:\Users\ensi\Desktop\python ai\ai_jobs_cleaned")

LIST_FIELDS = ["responsibilities", "basic_qualifications", "preferred_qualifications"]

stats = {
    "had_empty_bullets"    : 0,
    "had_duplicate_bullets": 0,
    "had_whitespace_issues": 0,
    "had_wrong_type"       : 0,
}

for fp in ORIGINAL_FOLDER.glob("*.yaml"):
    orig = yaml.safe_load(fp.read_text(encoding="utf-8", errors="replace"))

    for field in LIST_FIELDS:
        val = orig.get(field)

        # wrong type (not a list)
        if val is not None and not isinstance(val, list):
            stats["had_wrong_type"] += 1

        items = val if isinstance(val, list) else []

        for item in items:
            if item is None or (isinstance(item, str) and not item.strip()):
                stats["had_empty_bullets"] += 1

            if isinstance(item, str):
                if item != item.strip() or "  " in item:
                    stats["had_whitespace_issues"] += 1

        # duplicates
        clean_items = [str(i).strip() for i in items if i is not None and str(i).strip()]
        if len(clean_items) != len(set(clean_items)):
            stats["had_duplicate_bullets"] += 1

for k, v in stats.items():
    print(f"{k:<28}: {v}")

had_empty_bullets           : 0
had_duplicate_bullets       : 740
had_whitespace_issues       : 243
had_wrong_type              : 0
